# DISCRETE DIFFUSION FOR KEYWORD BASED REVIEW GENERATION

# Description

Our project will explore text generation (specifically reviews) given a prompt using MDLM and compare the results with traditional autoregressive models such as GPT-2. We will use the Yelp dataset for training and evaluation using task oriented metrics such as F1, ROUGE, and exact match with existing reviews.

Link to our MDLM model: https://huggingface.co/xenosaac/Sai_BaBa_MDLM/tree/main


In [1]:
import torch

mdlm_model_path = "mdlm_model.pth"
checkpoint = torch.load(mdlm_model_path)

In [2]:
# MUST RUN: Setup: dependencies, reproducibility, and Yelp dataset load

import os
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch

from datasets import load_dataset

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


# to be used later in the pipeline)
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 175
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Training Device selection (CUDA > Apple Silicon MPS > CPU)
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("using Nvidia Cuda GPU")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("using Apple Silicon")
else:
    DEVICE = "cpu"
    print("using CPU")

# Yelp dataset (HuggingFace) for review text generation.
# We use yelp_polarity because it provides large-scale review text plus a simple binary label (0=neg, 1=pos),
# which we can optionally use for conditioning / CFG later.
YELP_DATASET_NAME = "yelp_polarity"
raw_datasets = load_dataset(YELP_DATASET_NAME)

# The Review Text Format: raw_datasets["train"][i]["text"]



using Nvidia Cuda GPU


In [3]:
# Dataset Sanity Check, should run with no issue after executing cell above.

from collections import Counter

print("Dataset name:", YELP_DATASET_NAME)
print("Splits:", list(raw_datasets.keys()))
print("Train size:", len(raw_datasets["train"]))
print("Test size:", len(raw_datasets["test"]))
print("Columns:", raw_datasets["train"].column_names)

# Show a few examples
for i in range(3):
    ex = raw_datasets["train"][i]
    print("\n--- Example", i, "---")
    print("label:", ex.get("label"))
    print("text (first 300 chars):", ex.get("text", "")[:300].replace("\n", " "))

# Label distribution (sample a subset for speed)
SAMPLE_N = 10000
labels = [raw_datasets["train"][i]["label"] for i in range(min(SAMPLE_N, len(raw_datasets["train"])))]
label_counts = Counter(labels)
print("\nLabel distribution (sampled):", dict(label_counts))

# Basic length statistics (chars + word tokens)
def simple_word_count(s: str) -> int:
    return len(nltk.word_tokenize(s))

texts = [raw_datasets["train"][i]["text"] for i in range(min(2000, len(raw_datasets["train"])))]
char_lens = np.array([len(t) for t in texts], dtype=np.int32)
word_lens = np.array([simple_word_count(t) for t in texts], dtype=np.int32)

def describe(arr: np.ndarray, name: str):
    print(f"{name}: mean={arr.mean():.1f}, median={np.median(arr):.1f}, "
          f"p90={np.percentile(arr, 90):.1f}, p95={np.percentile(arr, 95):.1f}, max={arr.max()}")

print("\nLength stats on 2,000 train examples:")
describe(char_lens, "chars")
describe(word_lens, "word_tokens")

# Label meaning reminder for yelp_polarity (common convention)
if YELP_DATASET_NAME == "yelp_polarity":
    print("\nLabel meaning (yelp_polarity): 0 = negative, 1 = positive")


Dataset name: yelp_polarity
Splits: ['train', 'test']
Train size: 560000
Test size: 38000
Columns: ['text', 'label']

--- Example 0 ---
label: 0
text (first 300 chars): Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has ti

--- Example 1 ---
label: 1
text (first 300 chars): Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff, that I found out I have fibroids. He explores all options with

--- Example 2 ---
label: 0
text (first 300 chars): I don't know what Dr. Goldberg was like before  moving to Arizona, but let me tell you, STAY AWAY from this doctor and this o

In [4]:
# Minimal preprocessing utilities
# This block defines lightweight text normalization-
# -and length-clipping utilities (whitespace cleanup + safe truncation)
# to standardize Yelp reviews and control extreme outliers before-
# -keyword extraction and model tokenization.

# tldr Data normalization and standardization.
import re
import unicodedata

# Start conservative; Can tune later.
MAX_CHARS = 2000
MAX_WORD_TOKENS = 256
MIN_WORD_TOKENS = 20
MAX_WORD_TOKENS_FILTER = 200

def normalize_text(text: str) -> str:
    """
    Minimal, safe normalization:
    - Unicode normalize (NFKC)
    - Standardize whitespace
    - Strip leading/trailing spaces
    NOTE: We intentionally do NOT lowercase by default (can be toggled later).
    """
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clip_by_chars(text: str, max_chars: int = MAX_CHARS) -> str:
    """Clip very long reviews by character count (fast, tokenizer-independent)."""
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0].strip()  # avoid cutting mid-word if possible


def count_words_nltk(text: str) -> int:
    """Count word tokens using NLTK (used for quick stats / filtering)."""
    return len(nltk.word_tokenize(text))


def clip_by_word_tokens(text: str, max_tokens: int = MAX_WORD_TOKENS) -> str:
    """
    Clip by NLTK word tokens as a proxy length control.
    This is temporary until we finalize the model tokenizer (likely subword/BPE).
    """
    tokens = nltk.word_tokenize(text)
    if len(tokens) <= max_tokens:
        return text
    return " ".join(tokens[:max_tokens])


def preprocess_review(text: str) -> str:
    """One-stop minimal preprocessing for Yelp review text."""
    text = normalize_text(text)
    text = clip_by_chars(text, MAX_CHARS)
    text = clip_by_word_tokens(text, MAX_WORD_TOKENS)
    return text

def passes_length_filter(
    text: str,
    min_tokens: int = MIN_WORD_TOKENS,
    max_tokens: int = MAX_WORD_TOKENS_FILTER
) -> bool:
    """
    Returns True if a preprocessed review has an acceptable length.
    Used to remove reviews that are too short or
    too long for diffusion training, before keyword extraction.
    """
    n_tokens = count_words_nltk(text)
    return min_tokens <= n_tokens <= max_tokens

# Quick spot-check on a few samples
for i in range(3):
    raw_text = raw_datasets["train"][i]["text"]
    proc_text = preprocess_review(raw_text)
    print(f"\n--- Preprocess check {i} ---")
    print("raw chars:", len(raw_text), "| processed chars:", len(proc_text))
    print("raw preview:", raw_text[:140].replace("\n", " "))
    print("proc preview:", proc_text[:140].replace("\n", " "))



--- Preprocess check 0 ---
raw chars: 643 | processed chars: 636
raw preview: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- g
proc preview: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- g

--- Preprocess check 1 ---
raw chars: 495 | processed chars: 495
raw preview: Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years 
proc preview: Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years 

--- Preprocess check 2 ---
raw chars: 1143 | processed chars: 1142
raw preview: I don't know what Dr. Goldberg was like before  moving to Arizona, but let me tell you, STAY AWAY from this doctor and this office. I was go
proc preview: I don't know w

In [5]:
# Keyword extraction pipeline (KeyBERT first, RAKE optional)
# This block extracts a small set of representative keywords from each Yelp review,
# which we will later use as “labels” / prompt conditions for prompted review generation.
# We start with KeyBERT (semantic, embedding-based keywords)
# and keep RAKE as an optional lightweight fallback for comparison or debugging.
# Install KeyBERT if needed
from keybert import KeyBERT

# Optional RAKE
USE_RAKE = False
if USE_RAKE:
    from rake_nltk import Rake

# Initialize keyword models
kw_model = KeyBERT(model="all-MiniLM-L6-v2")

if USE_RAKE:
    rake_model = Rake()

def extract_keywords_keybert(text: str, top_k: int = 8):
    """
    Returns a list of keyword strings using KeyBERT.
    We remove very short tokens and keep unigrams/bigrams for simplicity.
    """
    text = preprocess_review(text)
    if not text:
        return []
    # KeyBERT returns list of (keyword, score)
    kws = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_k
    )
    # Keep only the keyword strings
    return [k for (k, _) in kws]

def extract_keywords_rake(text: str, top_k: int = 8):
    """
    Optional RAKE keyword extraction.
    Returns a list of keyword strings.
    """
    text = preprocess_review(text)
    if not text:
        return []
    rake_model.extract_keywords_from_text(text)
    return rake_model.get_ranked_phrases()[:top_k]

# Run keyword extraction on a small batch for sanity check
KW_SAMPLE_N = 200
examples = [raw_datasets["train"][i] for i in range(KW_SAMPLE_N)]

rows = []
for ex in tqdm(examples, desc="Extracting keywords"):
    text = ex["text"]
    label = ex["label"]
    kb_kws = extract_keywords_keybert(text, top_k=8)
    row = {
        "label": label,
        "text_preview": preprocess_review(text)[:200],
        "keybert_keywords": kb_kws,
    }
    if USE_RAKE:
        row["rake_keywords"] = extract_keywords_rake(text, top_k=8)
    rows.append(row)

kw_df = pd.DataFrame(rows)

# Show a few results
display(kw_df.head(5))


Extracting keywords:   0%|          | 0/200 [00:00<?, ?it/s]

,label,text_preview,keybert_keywords
0,0,"Unfortunately, the frustration of being Dr. Go...","[goldberg patient, dr goldberg, problem doctor..."
1,1,Been going to Dr. Goldberg for over 10 years. ...,"[dr goldberg, fibroids explores, patients, pat..."
2,0,I don't know what Dr. Goldberg was like before...,"[refills patients, doctor money, patients fina..."
3,0,I'm writing this review to give you a heads up...,"[health insurance, insurance cover, cover dr, ..."
4,1,All the food is great here. But the best thing...,"[pittsburgh, cajun best, wings nthe, dream pit..."


In [6]:
# Prompt/condition format
# This block defines a consistent way to convert extracted keywords
# (and an optional sentiment label) into a natural-language prompt.
# We will use this prompt as the conditioning input for prompted Yelp review generation,
# so the same format can be reused across SEDD training, sampling, and evaluation.
from typing import List, Optional

# Map Yelp Polarity labels to text (0=negative, 1=positive)
LABEL_TO_SENTIMENT = {0: "negative", 1: "positive"}

def clean_keyword(kw: str) -> str:
    """Light cleanup to keep prompts readable and consistent."""
    kw = normalize_text(kw)
    kw = kw.strip(" ,;:|")
    return kw

def build_prompt(
    keywords: List[str],
    label: Optional[int] = None,
    max_keywords: int = 6,
    include_sentiment: bool = True,
    natural_style: bool = True,
) -> str:
    """
    Build a prompt string used to condition generation.
    Examples:
      - "negative taste korean bbq service"
      - "PROMPT: negative; keywords=bbq, dry, salty"
    """
    kws = [clean_keyword(k) for k in keywords if k and isinstance(k, str)]
    # Deduplicate while preserving order
    seen = set()
    kws = [k for k in kws if not (k in seen or seen.add(k))]
    kws = kws[:max_keywords]

    sentiment = None
    if include_sentiment and label is not None and label in LABEL_TO_SENTIMENT:
        sentiment = LABEL_TO_SENTIMENT[label]

    if natural_style:
        parts = []
        if sentiment is not None:
            parts.append(sentiment)
        parts.extend(kws)
        return " ".join(parts).strip()
    else:
        # More explicit structured format if you prefer
        sent_part = f"sentiment={sentiment}" if sentiment is not None else "sentiment=<NULL>"
        kw_part = ", ".join(kws) if kws else "<NULL>"
        return f"PROMPT: {sent_part}; keywords={kw_part}"

def format_training_text(prompt: str, review_text: str) -> str:
    """
    Optional helper to create a single serialized string for debugging/logging.
    (MDLM pipelines may use separate fields; this is just convenient for inspection.)
    """
    prompt = normalize_text(prompt)
    review_text = preprocess_review(review_text)
    return f"PROMPT: {prompt}\nREVIEW: {review_text}"

# Sanity-check: build prompts for a few examples using KeyBERT keywords from our earlier dataframe
for i in range(3):
    ex = raw_datasets["train"][i]
    # Extract keywords on-the-fly for the check (for speed later, you'll cache keywords)
    kws = extract_keywords_keybert(ex["text"], top_k=8)
    prompt = build_prompt(kws, label=ex["label"], max_keywords=6, include_sentiment=True, natural_style=True)
    print("\n--- Prompt check", i, "---")
    print("Prompt:", prompt)
    print("Serialized preview:\n", format_training_text(prompt, ex["text"])[:350])



--- Prompt check 0 ---
Prompt: negative goldberg patient dr goldberg problem doctors doctors nyc doctor terrible goldberg
Serialized preview:
 PROMPT: negative goldberg patient dr goldberg problem doctors doctors nyc doctor terrible goldberg
REVIEW: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually takes 2 

--- Prompt check 1 ---
Prompt: positive dr goldberg fibroids explores patients patients started 1st patients fibroids
Serialized preview:
 PROMPT: positive dr goldberg fibroids explores patients patients started 1st patients fibroids
REVIEW: Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff, that I fo

--- Prompt ch

In [7]:
# Keyword caching (so KeyBERT doesn’t bottleneck everything)

"""
NOTE: That this is no longer used as, after testing, we found out that it wasn't a bottleneck
"""

import os
import json
import hashlib
from tqdm import tqdm

# store cache files
CACHE_DIR = "./cache"
os.makedirs(CACHE_DIR, exist_ok=True)

KEYWORD_CACHE_PATH = os.path.join(CACHE_DIR, "keyword_cache.json")

def load_keyword_cache(path=KEYWORD_CACHE_PATH):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_keyword_cache(cache, path=KEYWORD_CACHE_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

keyword_cache = load_keyword_cache()
print(f"Loaded {len(keyword_cache)} cached keyword entries")

def text_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()

Loaded 0 cached keyword entries


In [8]:
def get_cached_keywords(text: str, top_k: int = 8, use_rake: bool = False):
    text = preprocess_review(text)
    hid = text_hash(text)

    if hid in keyword_cache:
        return keyword_cache[hid]

    if use_rake:
        keywords = extract_keywords_rake(text, top_k=top_k)
    else:
        keywords = extract_keywords_keybert(text, top_k=top_k)

    keyword_cache[hid] = keywords

    return keywords


def flush_keyword_cache():
    save_keyword_cache(keyword_cache)
    print(f"Saved {len(keyword_cache)} keyword entries to {KEYWORD_CACHE_PATH}")

In [9]:
# Build the diffusion/SEDD -> Now MDLM training dataset object (NO training yet)

# This block constructs a dataset wrapper that returns:
#   - processed review text
#   - extracted keywords
#   - a formatted prompt string (optionally including sentiment)
#   - a CFG-ready version of the prompt via condition dropout (prompt -> <NULL>)
# No model training happens here.

from dataclasses import dataclass
from typing import Dict, Any, Tuple, List, Optional

# CFG / conditioning settings
CFG_DROP_PROB = 0.15  # probability to drop prompt during training (classifier-free guidance)
MAX_KEYWORDS = 6  # how many keywords to keep in the prompt
INCLUDE_SENTIMENT = True  # include yelp label as "positive/negative" in the prompt
NULL_PROMPT = "<NULL>"  # null condition token/string

# Keyword extraction settings
KEYWORD_TOP_K = 8 # raw keywords extracted before truncating to MAX_KEYWORDS

# How many examples to process for now (set None for full split; keep small while iterating)
MAX_EXAMPLES_PER_SPLIT = 560000 # increase later / set to None once stable


@dataclass
class PromptedExample:
    prompt: str
    prompt_dropped: str
    review: str
    label: int
    keywords: List[str]


class YelpPromptedDataset(torch.utils.data.Dataset):
    """
    Wraps a HuggingFace dataset split and produces prompted generation examples.
    Returned dict fields:
      - "prompt": conditioning string (natural language)
      - "prompt_dropped": prompt after CFG dropout (<NULL> sometimes)
      - "review": processed review text (target text)
      - "label": original label (e.g., 0/1)
      - "keywords": extracted keyword list (for debugging / evaluation)
    """
    def __init__(
        self,
        hf_split,
        max_examples: Optional[int] = None,
        cfg_drop_prob: float = 0.0,
        include_sentiment: bool = True,
        max_keywords: int = 6,
        keyword_top_k: int = 8,
        use_rake: bool = False,
        seed: int = 175,
    ):
        self.split = hf_split
        self.cfg_drop_prob = cfg_drop_prob
        self.include_sentiment = include_sentiment
        self.max_keywords = max_keywords
        self.keyword_top_k = keyword_top_k
        self.use_rake = use_rake

        self.rng = random.Random(seed)

        self.n = len(self.split) if max_examples is None else min(max_examples, len(self.split))

    def __len__(self) -> int:
        return self.n

    def _extract_keywords(self, text: str) -> List[str]:
        return get_cached_keywords(
            text,
            top_k=self.keyword_top_k,
            use_rake=self.use_rake,
        )

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.split[idx]
        label = int(ex["label"])
        review = preprocess_review(ex["text"])

        keywords = self._extract_keywords(review)
        prompt = build_prompt(
            keywords=keywords,
            label=label,
            max_keywords=self.max_keywords,
            include_sentiment=self.include_sentiment,
            natural_style=True,
        )

        # CFG condition dropout: sometimes replace prompt with NULL to train unconditional branch
        if self.cfg_drop_prob > 0 and self.rng.random() < self.cfg_drop_prob:
            prompt_dropped = NULL_PROMPT
        else:
            prompt_dropped = prompt

        return {
            "prompt": prompt,
            "prompt_dropped": prompt_dropped,
            "review": review,
            "label": label,
            "keywords": keywords,
        }


def build_splits_for_sedd(
    raw_datasets,
    max_train_examples: Optional[int] = None,
    max_test_examples: Optional[int] = None,
    cfg_drop_prob: float = 0.0,
    include_sentiment: bool = True,
    max_keywords: int = 6,
    keyword_top_k: int = 8,
    use_rake: bool = False,
    seed: int = 175,
):
    train_ds = YelpPromptedDataset(
        raw_datasets["train"],
        max_examples=max_train_examples,
        cfg_drop_prob=cfg_drop_prob,
        include_sentiment=include_sentiment,
        max_keywords=max_keywords,
        keyword_top_k=keyword_top_k,
        use_rake=use_rake,
        seed=seed,
    )
    test_ds = YelpPromptedDataset(
        raw_datasets["test"],
        max_examples=max_test_examples,
        cfg_drop_prob=0.0,  # no dropout for evaluation split object
        include_sentiment=include_sentiment,
        max_keywords=max_keywords,
        keyword_top_k=keyword_top_k,
        use_rake=use_rake,
        seed=seed,
    )
    return train_ds, test_ds


train_prompted_ds, test_prompted_ds = build_splits_for_sedd(
    raw_datasets=raw_datasets,
    max_train_examples=560000,
    max_test_examples=38000,
    cfg_drop_prob=CFG_DROP_PROB,
    include_sentiment=INCLUDE_SENTIMENT,
    max_keywords=MAX_KEYWORDS,
    keyword_top_k=KEYWORD_TOP_K,
    use_rake=False,
    seed=SEED,
)

print("Built datasets:")
print("  train_prompted_ds size:", len(train_prompted_ds))
print("  test_prompted_ds size:", len(test_prompted_ds))

# Inspect a few samples
for i in range(3):
    item = train_prompted_ds[i]
    print(f"\n--- Prompted sample {i} ---")
    print("label:", item["label"])
    print("keywords:", item["keywords"][:8])
    print("prompt:", item["prompt"])
    print("prompt_dropped:", item["prompt_dropped"])
    print("review preview:", item["review"][:220])


Built datasets:
  train_prompted_ds size: 560000
  test_prompted_ds size: 38000

--- Prompted sample 0 ---
label: 0
keywords: ['goldberg patient', 'dr goldberg', 'problem doctors', 'doctors nyc', 'doctor terrible', 'goldberg', 'patients medical', 'doctor']
prompt: negative goldberg patient dr goldberg problem doctors doctors nyc doctor terrible goldberg
prompt_dropped: negative goldberg patient dr goldberg problem doctors doctors nyc doctor terrible goldberg
review preview: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the pho

--- Prompted sample 1 ---
label: 1
keywords: ['dr goldberg', 'fibroids explores', 'patients', 'patients started', '1st patients', 'fibroids', 'gyn dr', 'markoff fibroids']
prompt: positive dr goldberg fibroids explores patients patients started 1st patients fibroids
prompt_dropped: <NULL>
review pr

In [10]:
# Tokenization decision for the diffusion model (NLTK)
# This block compares two tokenization options at a small scale WITHOUT triggering KeyBERT.
#   (A) NLTK word tokenization + custom vocab (simple, but large vocab / more UNKs)
#   (B) Subword/BPE tokenizer from HuggingFace (recommended for most diffusion/MDLM codebases)
# We do NOT train anything here — only measure token length stats and OOV/UNK behavior.

"""
NOTE: We are now using a gpt2 tokenizer as we realized that gpt2's bpe tokenization works better
"""

from collections import Counter
from typing import List

# IMPORTANT: sample directly from raw_datasets to avoid running KeyBERT inside train_prompted_ds
TOKEN_SAMPLE_N = 2000  # keep small for fast iteration; increase later if needed

sample_texts = [
    preprocess_review(raw_datasets["train"][i]["text"])
    for i in range(min(TOKEN_SAMPLE_N, len(raw_datasets["train"])))
]

# -----------------------------
# Option A: NLTK word tokenizer
# -----------------------------
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[MASK]", "[BOS]", "[EOS]", "[SEP]"]
VOCAB_SIZE = 20000  # smaller vocab for faster stats (adjust later)

def nltk_tokenize(text: str) -> List[str]:
    return nltk.word_tokenize(text)

# Build vocab from sampled data
word_counter = Counter()
for t in tqdm(sample_texts, desc="Building NLTK vocab"):
    word_counter.update([w.lower() for w in nltk_tokenize(t)])

most_common_words = [w for w, _ in word_counter.most_common(VOCAB_SIZE - len(SPECIAL_TOKENS))]
nltk_vocab = SPECIAL_TOKENS + most_common_words
nltk_stoi = {tok: i for i, tok in enumerate(nltk_vocab)}
UNK_ID = nltk_stoi["[UNK]"]

def nltk_encode(text: str) -> List[int]:
    toks = [w.lower() for w in nltk_tokenize(text)]
    return [nltk_stoi.get(tok, UNK_ID) for tok in toks]

# Compute length + UNK stats
nltk_lens = []
nltk_unk_rates = []
for t in tqdm(sample_texts, desc="NLTK encode stats"):
    ids = nltk_encode(t)
    nltk_lens.append(len(ids))
    if len(ids) == 0:
        nltk_unk_rates.append(0.0)
    else:
        nltk_unk_rates.append(sum(1 for x in ids if x == UNK_ID) / len(ids))

nltk_lens = np.array(nltk_lens)
nltk_unk_rates = np.array(nltk_unk_rates)

print("\nNLTK word tokenizer + custom vocab for SEDD")
print("Vocab size:", len(nltk_vocab))
print(f"Token length: mean={nltk_lens.mean():.1f}, median={np.median(nltk_lens):.1f}, "
      f"p90={np.percentile(nltk_lens, 90):.1f}, p95={np.percentile(nltk_lens, 95):.1f}, max={nltk_lens.max()}")
print(f"UNK rate: mean={nltk_unk_rates.mean():.4f}, p95={np.percentile(nltk_unk_rates, 95):.4f}, max={nltk_unk_rates.max():.4f}")

# ------------------------------------------
# Option B: Subword tokenizer (HuggingFace)
# ------------------------------------------
HF_TOKENIZER_NAME = "gpt2"

hf_tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_NAME)
# GPT2 has no pad token by default; set it for batching/length stats
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token

hf_lens = []
for t in tqdm(sample_texts, desc=f"HF({HF_TOKENIZER_NAME}) encode stats"):
    ids = hf_tokenizer.encode(t, add_special_tokens=False)
    hf_lens.append(len(ids))

hf_lens = np.array(hf_lens)

print(f"\nHuggingFace subword tokenizer (for GPT 2 comparison test)({HF_TOKENIZER_NAME})")
print("Vocab size:", hf_tokenizer.vocab_size)
print(f"Token length: mean={hf_lens.mean():.1f}, median={np.median(hf_lens):.1f}, "
      f"p90={np.percentile(hf_lens, 90):.1f}, p95={np.percentile(hf_lens, 95):.1f}, max={hf_lens.max()}")



NLTK encode stats: 100%|██████████| 2000/2000 [00:00<00:00, 2173.87it/s]



NLTK word tokenizer + custom vocab for SEDD
Vocab size: 14686
Token length: mean=130.8, median=117.0, p90=256.0, p95=256.0, max=257
UNK rate: mean=0.0000, p95=0.0000, max=0.0000


HF(gpt2) encode stats: 100%|██████████| 2000/2000 [00:00<00:00, 3707.04it/s]


HuggingFace subword tokenizer (for GPT 2 comparison test)(gpt2)
Vocab size: 50257
Token length: mean=145.6, median=127.0, p90=286.0, p95=295.0, max=380


In [11]:
# DataLoader + batch inspection

# This block builds a minimal DataLoader for our prompted examples and inspects one batch.
# No training happens here.

from typing import List, Dict, Any
from torch.utils.data import DataLoader

BATCH_SIZE = 8

def prompted_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Keep everything as Python strings/lists for now.
    # (Later, when we align with MDLM tokenizer, this collator will tokenize and pad.)
    return {
        "prompt": [b["prompt"] for b in batch],
        "prompt_dropped": [b["prompt_dropped"] for b in batch],
        "review": [b["review"] for b in batch],
        "label": torch.tensor([b["label"] for b in batch], dtype=torch.long),
        "keywords": [b["keywords"] for b in batch],
    }

train_loader = DataLoader(
    train_prompted_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # keep 0 in Colab for fewer surprises; increase later if stable
    collate_fn=prompted_collate_fn,
)

batch = next(iter(train_loader))

print("Batch keys:", list(batch.keys()))
print("label shape:", batch["label"].shape)

print("\n--- Batch sample[0] ---")
print("label:", int(batch["label"][0]))
print("prompt:", batch["prompt"][0])
print("prompt_dropped:", batch["prompt_dropped"][0])
print("keywords:", batch["keywords"][0])
print("review preview:", batch["review"][0][:300])

# Check CFG dropout frequency in this batch
drop_count = sum(1 for p in batch["prompt_dropped"] if p == NULL_PROMPT)
print(f"\nCFG drop count in batch: {drop_count}/{BATCH_SIZE} (CFG_DROP_PROB={CFG_DROP_PROB})")


Batch keys: ['prompt', 'prompt_dropped', 'review', 'label', 'keywords']
label shape: torch.Size([8])

--- Batch sample[0] ---
label: 1
prompt: positive casino disneyland mandalay bay free rooms disneyland disneyland typically stay mandalay
prompt_dropped: positive casino disneyland mandalay bay free rooms disneyland disneyland typically stay mandalay
keywords: ['casino disneyland', 'mandalay bay', 'free rooms', 'disneyland', 'disneyland typically', 'stay mandalay', 'hotel', 'mandalay']
review preview: I usually stay at Mandalay Bay but I was offered free rooms at the newly renovated Delano. Though my offer stated that a particular night was free, the rep at MLife said I would have to pay $89 because it was the weekend of the iheartradio concert and rates were subject to change. I agreed since the

CFG drop count in batch: 1/8 (CFG_DROP_PROB=0.15)


In [12]:
import time

print("Trying to fetch one batch...")
t0 = time.time()
b = next(iter(train_loader))
print("Got batch in", time.time() - t0, "seconds")
print({k: (v.shape if hasattr(v, "shape") else type(v)) for k,v in b.items()})

Trying to fetch one batch...
Got batch in 0.2709980010986328 seconds
{'prompt': <class 'list'>, 'prompt_dropped': <class 'list'>, 'review': <class 'list'>, 'label': torch.Size([8]), 'keywords': <class 'list'>}


In [13]:
# GPT-2 SANITY CHECKER / BASELINE
# This block establishes baseline outputs using pretrained GPT-2 with the same prompt format
# as your SEDD pipeline. Use this to understand what "good" autoregressive outputs look like.

print("=" * 80)
print("GPT-2 BASELINE SANITY CHECKER")
print("=" * 80)

# Configuration
GPT2_MODEL_NAME = "gpt2"  # Can also try "gpt2-medium", "gpt2-large" for better quality
MAX_NEW_TOKENS = 150  # How many tokens to generate per review
NUM_SAMPLES = 2  # Number of examples to generate
GENERATION_SEED = 42  # For reproducible generation

# Load GPT-2 model and tokenizer
print(f"\nLoading GPT-2 model: {GPT2_MODEL_NAME}")
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME)

# Set pad token (GPT-2 doesn't have one by default)
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
    gpt2_model.config.pad_token_id = gpt2_model.config.eos_token_id

# Move model to device
gpt2_model = gpt2_model.to(DEVICE)
gpt2_model.eval()

print(f"Model loaded on {DEVICE}")
print(f"Model parameters: {sum(p.numel() for p in gpt2_model.parameters()) / 1e6:.1f}M")

# Function to generate review from prompt
def generate_review_gpt2(
    prompt: str,
    max_new_tokens: int = 150,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
    num_return_sequences: int = 1,
    do_sample: bool = True,
):
    """
    Generate review text from a prompt using GPT-2.

    Args:
        prompt: The conditioning prompt (e.g., "negative korean bbq dry service")
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (higher = more random)
        top_k: Top-k sampling parameter
        top_p: Nucleus sampling parameter
        num_return_sequences: Number of sequences to generate
        do_sample: Whether to use sampling (False = greedy)

    Returns:
        List of generated review texts
    """
    # Format prompt for GPT-2 (add explicit separator)
    formatted_prompt = f"{prompt}\n\nReview:"

    # Tokenize
    inputs = gpt2_tokenizer(
        formatted_prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(DEVICE)

    # Generate
    with torch.no_grad():
        outputs = gpt2_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            do_sample=do_sample,
            pad_token_id=gpt2_tokenizer.pad_token_id,
            eos_token_id=gpt2_tokenizer.eos_token_id,
        )

    # Decode
    generated_texts = []
    for output in outputs:
        full_text = gpt2_tokenizer.decode(output, skip_special_tokens=True)
        # Remove the prompt part
        review_text = full_text.split("Review:", 1)[-1].strip()
        generated_texts.append(review_text)

    return generated_texts


# Test on sample prompts from your dataset
print("\n" + "=" * 80)
print("ZERO-SHOT GENERATION SAMPLES (Pretrained GPT-2)")
print("=" * 80)

# Set seed for reproducibility
torch.manual_seed(GENERATION_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GENERATION_SEED)

# Generate from test set examples
for i in range(NUM_SAMPLES):
    print(f"\n{'=' * 80}")
    print(f"SAMPLE {i+1}/{NUM_SAMPLES}")
    print(f"{'=' * 80}")

    # Get a real example from your dataset
    sample = test_prompted_ds[i]
    prompt = sample["prompt"]
    true_review = sample["review"]
    label = "positive" if sample["label"] == 1 else "negative"

    print(f"\nPrompt: {prompt}")
    print(f"\nTrue Label: {label}")
    print(f"\nTrue Review (first 300 chars):\n{true_review[:300]}...")

    # Generate with different decoding strategies
    print(f"\n{'-' * 80}")
    print("GPT-2 GENERATION (Temperature=0.8, Top-p=0.95):")
    print(f"{'-' * 80}")

    generated = generate_review_gpt2(
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        do_sample=True,
    )
    print(generated[0])

    print(f"\n{'-' * 80}")
    print("GPT-2 GENERATION (Greedy Decoding):")
    print(f"{'-' * 80}")

    generated_greedy = generate_review_gpt2(
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )
    print(generated_greedy[0])


# Compare prompts with different sentiments
print("\n" + "=" * 80)
print("SENTIMENT CONDITIONING TEST")
print("=" * 80)

# Create positive and negative prompts with same keywords
test_keywords = ["food", "service", "atmosphere"]

for sentiment_label in [0, 1]:
    sentiment = LABEL_TO_SENTIMENT[sentiment_label]
    test_prompt = build_prompt(
        keywords=test_keywords,
        label=sentiment_label,
        max_keywords=6,
        include_sentiment=True,
        natural_style=True
    )

    print(f"\n{'-' * 80}")
    print(f"Prompt ({sentiment}): {test_prompt}")
    print(f"{'-' * 80}")

    gen = generate_review_gpt2(
        prompt=test_prompt,
        max_new_tokens=100,
        temperature=0.8,
        do_sample=True,
    )
    print(gen[0])


print("\n" + "=" * 80)
print("GPT-2 SANITY CHECK COMPLETE")
print("=" * 80)

GPT-2 BASELINE SANITY CHECKER

Loading GPT-2 model: gpt2
Model loaded on cuda
Model parameters: 124.4M

ZERO-SHOT GENERATION SAMPLES (Pretrained GPT-2)

SAMPLE 1/2

Prompt: positive tire service service road renovated waiting complaints service service prices getting tire

True Label: positive

True Review (first 300 chars):
Contrary to other reviews, I have zero complaints about the service or the prices. I have been getting tire service here for the past 5 years now, and compared to my experience with places like Pep Boys, these guys are experienced and know what they're doing. \nAlso, this is one place that I do not ...

--------------------------------------------------------------------------------
GPT-2 GENERATION (Temperature=0.8, Top-p=0.95):
--------------------------------------------------------------------------------


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Toyota Corolla

--------------------------------------------------------------------------------
GPT-2 GENERATION (Greedy Decoding):
--------------------------------------------------------------------------------
The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service road. The new tire service road is a great addition to the existing service

SAMPLE 2/2

Prompt: negative pr

In [14]:
"""
NOTE: This is commented out as it would take too long to run, but it was our main testing comparison
      Not the Zero-Shot that was ran above it (it was just ran because it takes no time)
"""

# # FINE-TUNE GPT-2 ON YELP DATA
# # This block shows how to fine-tune GPT-2 on your prompted Yelp dataset
# # for a stronger baseline. Skip this if you just want zero-shot testing.




# print("=" * 80)
# print("GPT-2 FINE-TUNING ON YELP REVIEWS")
# print("=" * 80)

# # Fine-tuning configuration
# FINETUNE_EPOCHS = 3
# FINETUNE_BATCH_SIZE = 8
# FINETUNE_LR = 5e-5
# FINETUNE_MAX_LENGTH = 256  # Max sequence length for training

# import os
# FINETUNE_OUTPUT_DIR = os.path.join(os.getcwd(), "gpt2_yelp_finetuned")


# # Create a tokenized dataset for GPT-2 training
# class GPT2YelpDataset(torch.utils.data.Dataset):
#     """
#     Dataset that formats prompted reviews for GPT-2 causal LM training.
#     Format: "{prompt}\n\nReview: {review_text}<|endoftext|>"
#     """
#     def __init__(self, prompted_dataset, tokenizer, max_length=256):
#         self.prompted_ds = prompted_dataset
#         self.tokenizer = tokenizer
#         self.max_length = max_length

#     def __len__(self):
#         return len(self.prompted_ds)

#     def __getitem__(self, idx):
#         item = self.prompted_ds[idx]

#         # Format: prompt + review
#         # Use prompt_dropped to leverage CFG training data
#         prompt = item["prompt_dropped"]
#         review = item["review"]

#         # Create training text
#         if prompt == NULL_PROMPT:
#             # Unconditional generation (no prompt)
#             text = f"Review: {review}"
#         else:
#             # Conditional generation (with prompt)
#             text = f"{prompt}\n\nReview: {review}"

#         # Tokenize
#         encoding = self.tokenizer(
#             text,
#             truncation=True,
#             max_length=self.max_length,
#             padding="max_length",
#             return_tensors="pt"
#         )

#         # For causal LM, labels = input_ids
#         return {
#             "input_ids": encoding["input_ids"].squeeze(),
#             "attention_mask": encoding["attention_mask"].squeeze(),
#             "labels": encoding["input_ids"].squeeze(),
#         }


# # Build training dataset
# print("\nPreparing training data...")
# gpt2_train_dataset = GPT2YelpDataset(
#     prompted_dataset=train_prompted_ds,
#     tokenizer=gpt2_tokenizer,
#     max_length=FINETUNE_MAX_LENGTH
# )

# from torch.utils.data import Subset
# gpt2_train_dataset = Subset(gpt2_train_dataset, range(50000))

# gpt2_eval_dataset = GPT2YelpDataset(
#     prompted_dataset=test_prompted_ds,
#     tokenizer=gpt2_tokenizer,
#     max_length=FINETUNE_MAX_LENGTH
# )

# print(f"Training samples: {len(gpt2_train_dataset)}")
# print(f"Eval samples: {len(gpt2_eval_dataset)}")

# # Training arguments
# training_args = TrainingArguments(
#     output_dir=FINETUNE_OUTPUT_DIR,
#     num_train_epochs=FINETUNE_EPOCHS,
#     per_device_train_batch_size=FINETUNE_BATCH_SIZE,
#     per_device_eval_batch_size=FINETUNE_BATCH_SIZE,
#     learning_rate=FINETUNE_LR,
#     warmup_steps=100,
#     logging_steps=50,
#     eval_steps=200,
#     save_steps=600,
#     eval_strategy="steps",
#     save_total_limit=2,
#     load_best_model_at_end=True,
#     fp16=torch.cuda.is_available(),  # Use mixed precision if on GPU
#     report_to="none",  # Disable wandb/tensorboard for simplicity
#     seed=SEED,
# )

# # Initialize trainer
# print("\nInitializing trainer...")
# trainer = Trainer(
#     model=gpt2_model,
#     args=training_args,
#     train_dataset=gpt2_train_dataset,
#     eval_dataset=gpt2_eval_dataset,
#     processing_class=gpt2_tokenizer,
# )

# # Start training
# print("\n" + "=" * 80)
# print("STARTING FINE-TUNING...")
# print("=" * 80)
# print(f"Training for {FINETUNE_EPOCHS} epochs on {len(gpt2_train_dataset)} examples")
# print(f"Batch size: {FINETUNE_BATCH_SIZE}, Learning rate: {FINETUNE_LR}")
# print("=" * 80 + "\n")

# # Train
# trainer.train()

# print("\n" + "=" * 80)
# print("TRAINING COMPLETE!")
# print("=" * 80)
# trainer.save_model(FINETUNE_OUTPUT_DIR)
# gpt2_tokenizer.save_pretrained(FINETUNE_OUTPUT_DIR)
# print(f"\nModel saved to: {FINETUNE_OUTPUT_DIR}")

# # Evaluate on test set
# print("\nEvaluating on test set...")
# eval_results = trainer.evaluate()
# print(f"\nTest Loss: {eval_results['eval_loss']:.4f}")
# print(f"Test Perplexity: {np.exp(eval_results['eval_loss']):.2f}")

# # Test generation with fine-tuned model
# print("\n" + "=" * 80)
# print("TESTING FINE-TUNED MODEL GENERATION")
# print("=" * 80)

# for i in range(3):
#     sample = test_prompted_ds[i]
#     prompt = sample["prompt"]

#     print(f"\n{'=' * 80}")
#     print(f"FINE-TUNED SAMPLE {i+1}")
#     print(f"{'=' * 80}")
#     print(f"Prompt: {prompt}\n")

#     generated = generate_review_gpt2(
#         prompt=prompt,
#         max_new_tokens=120,
#         temperature=0.8,
#         do_sample=True,
#     )

#     print(f"Generated Review:\n{generated[0]}")
#     print(f"\nTrue Review:\n{sample['review'][:300]}...")

# print("\n" + "=" * 80)
# print("Fine-tuning complete! Compare these outputs with pretrained GPT-2.")
# print("=" * 80)

'\nNOTE: This is commented out as it would take too long to run, but it was our main testing comparison\n      Not the Zero-Shot that was ran above it (it was just ran because it takes no time)\n'

In [15]:
"""
NOTE: This is commented out because it is tied to the fine-tuned model which is also commented out
"""

# import numpy as np

# Store GPT-2 results for comparison table
# gpt2_loss = eval_results["eval_loss"]
# gpt2_perplexity = np.exp(gpt2_loss)

# print("GPT-2 Evaluation Results")
# print("------------------------")
# print("Loss:", gpt2_loss)
# print("Perplexity:", gpt2_perplexity)

'\nNOTE: This is commented out because it is tied to the fine-tuned model which is also commented out\n'

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer
import numpy as np
from tqdm.auto import tqdm
import math
import os


class AdaLN(nn.Module):
    def __init__(self, d_model, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(d_model, elementwise_affine=False)
        self.proj = nn.Linear(cond_dim, d_model * 2)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x, cond):
        x = self.norm(x)
        scale, shift = self.proj(cond).unsqueeze(1).chunk(2, dim=-1)
        return x * (1 + scale) + shift


class MDLMTransformer(nn.Module):
    def __init__(self, vocab_size=50259, d_model=768, nhead=8, num_layers=8,
                 dim_feedforward=3072, dropout=0.1, max_seq_len=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        self.cond_proj = nn.Linear(d_model, d_model)
        self.adaln = AdaLN(d_model, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output_layer = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                if module is self.adaln.proj:
                    continue
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, condition_tokens=None):
        batch_size, seq_len = x.shape
        device = x.device

        token_emb = self.token_embedding(x)
        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
        pos_emb = self.pos_embedding(positions)
        h = token_emb + pos_emb

        if condition_tokens is not None:
            cond_emb = self.token_embedding(condition_tokens)
            cond = self.cond_proj(cond_emb.mean(dim=1))
            h = self.adaln(h, cond)

        h = self.transformer(h)

        if condition_tokens is not None:
            h = self.adaln(h, cond)

        logits = self.output_layer(h)
        return logits


class MDLMYelpDataset(Dataset):
    def __init__(self, prompted_dataset, tokenizer, max_length=256, max_keyword_length=32):
        self.prompted_ds = prompted_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_keyword_length = max_keyword_length
        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id

    def __len__(self):
        return len(self.prompted_ds)

    def __getitem__(self, idx):
        item = self.prompted_ds[idx]
        review = item["review"]
        prompt = item["prompt_dropped"]

        review_encoding = self.tokenizer(
            review, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )

        if prompt == "<NULL>":
            keyword_ids = torch.full((self.max_keyword_length,), self.pad_token_id, dtype=torch.long)
        else:
            keyword_encoding = self.tokenizer(
                prompt, truncation=True, max_length=self.max_keyword_length,
                padding="max_length", return_tensors="pt"
            )
            keyword_ids = keyword_encoding["input_ids"].squeeze()

        return {
            "input_ids": review_encoding["input_ids"].squeeze(),
            "attention_mask": review_encoding["attention_mask"].squeeze(),
            "keyword_ids": keyword_ids,
            "label": item["label"],
        }


def get_mask_prob(t, schedule='cosine'):
    if schedule == 'cosine':
        s = 0.008
        return np.cos(((t + s) / (1 + s)) * np.pi / 2) ** 2
    else:
        return t


def create_masked_input(clean_tokens, mask_token_id, mask_prob, attention_mask=None):
    batch_size, seq_len = clean_tokens.shape
    device = clean_tokens.device
    random_mask = torch.rand(clean_tokens.shape, device=device)
    if isinstance(mask_prob, torch.Tensor) and mask_prob.dim() > 0:
        mask_prob = mask_prob.view(batch_size, 1)
    mask_indices = random_mask < mask_prob
    if attention_mask is not None:
        mask_indices = mask_indices & (attention_mask.bool())
    noisy_input = clean_tokens.clone()
    noisy_input[mask_indices] = mask_token_id
    return noisy_input, mask_indices


def train_step(model, batch, optimizer, mask_token_id, schedule='cosine', device='cuda'):
    model.train()
    clean_tokens = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    keyword_ids = batch["keyword_ids"].to(device)
    batch_size, seq_len = clean_tokens.shape

    t = torch.rand(batch_size, 1, device=device)
    mask_probs = torch.tensor(
        [get_mask_prob(t_i.item(), schedule) for t_i in t], device=device
    ).view(batch_size, 1)

    noisy_input, mask_indices = create_masked_input(
        clean_tokens, mask_token_id, mask_probs, attention_mask
    )

    logits = model(noisy_input, condition_tokens=keyword_ids)   # no t
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)), clean_tokens.view(-1), reduction='none'
    )
    loss = loss.view(batch_size, seq_len)
    masked_loss = (loss * mask_indices.float()).sum() / (mask_indices.sum() + 1e-8)

    optimizer.zero_grad()
    masked_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    with torch.no_grad():
        predictions = logits.argmax(dim=-1)
        correct = (predictions == clean_tokens) & mask_indices
        accuracy = correct.sum().float() / (mask_indices.sum() + 1e-8)

    return masked_loss.item(), accuracy.item()


@torch.no_grad()
def sample_mdlm(model, tokenizer, keywords=None, n_steps=150, guidance_scale=3.0,
                temperature=1.0, max_length=128, device='cuda'):

    model.eval()
    vocab_size = model.vocab_size
    mask_token_id = vocab_size - 1

    if keywords is None or keywords == "<NULL>":
        keyword_ids = torch.full((1, 32), tokenizer.pad_token_id, dtype=torch.long, device=device)
        use_cfg = False
    else:
        keyword_encoding = tokenizer(
            keywords,
            truncation=True,
            max_length=32,
            padding="max_length",
            return_tensors="pt"
        )
        keyword_ids = keyword_encoding["input_ids"].to(device)

        keyword_ids = torch.clamp(keyword_ids, max=vocab_size - 1)

        use_cfg = guidance_scale > 1.0

    mask_token_id = model.vocab_size - 1
    x = torch.full((1, max_length), mask_token_id, dtype=torch.long, device=device)
    null_keyword_ids = torch.full((1, 32), tokenizer.pad_token_id, dtype=torch.long, device=device)

    for step in range(n_steps):

        x = torch.clamp(x, max=vocab_size - 1)

        if use_cfg:
            logits_cond = model(x, condition_tokens=keyword_ids)
            logits_uncond = model(x, condition_tokens=null_keyword_ids)
            logits = logits_uncond + guidance_scale * (logits_cond - logits_uncond)
        else:
            logits = model(x, condition_tokens=keyword_ids)

        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)

        sampled_tokens = torch.multinomial(probs[0], num_samples=1).squeeze(-1)

        mask_positions = (x[0] == mask_token_id).nonzero(as_tuple=True)[0]

        if len(mask_positions) > 0:
            n_unmask = max(1, int(len(mask_positions) * (1.0 / (n_steps - step))))
            masked_probs = probs[0, mask_positions]
            max_probs, _ = masked_probs.max(dim=-1)

            _, confident_indices = torch.topk(
                max_probs,
                min(n_unmask, len(mask_positions))
            )

            positions_to_unmask = mask_positions[confident_indices]
            x[0, positions_to_unmask] = sampled_tokens[positions_to_unmask]

    return tokenizer.decode(x[0].cpu(), skip_special_tokens=True)

In [17]:
"""
NOTE: This is commented out because it is the training process for the MDLM model
"""

# def quick_test_mdlm():

#     print("="*80)
#     print("MDLM  TEST ")
#     print("="*80)


#     print("1. Setting up tokenizer...")
#     tokenizer = AutoTokenizer.from_pretrained("gpt2")
#     tokenizer.add_special_tokens({'mask_token': '[MASK]', 'pad_token': '<PAD>'})
#     vocab_size = len(tokenizer)
#     print(f"   ✓ Vocab size: {vocab_size}")
#     print(f"   ✓ [MASK] ID: {tokenizer.mask_token_id}")


#     print("\n2. Creating datasets for test...")

#     full_train_dataset = MDLMYelpDataset(
#         train_prompted_ds, tokenizer, max_length=128, max_keyword_length=32
#     )
#     full_val_dataset = MDLMYelpDataset(
#         test_prompted_ds, tokenizer, max_length=128, max_keyword_length=32
#     )

#     train_subset_size = 560000
#     val_subset_size   = 38000

#     train_indices = list(range(min(train_subset_size, len(full_train_dataset))))
#     val_indices   = list(range(min(val_subset_size,   len(full_val_dataset))))

#     train_dataset = Subset(full_train_dataset, train_indices)
#     val_dataset   = Subset(full_val_dataset,   val_indices)

#     print(f"   ✓ Train: {len(train_dataset)} samples")
#     print(f"   ✓ Val:   {len(val_dataset)} samples")

#     train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0, pin_memory=True)
#     val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
#     print(f"   ✓ Train batches: {len(train_loader)}")
#     print(f"   ✓ Val batches:   {len(val_loader)}")


#     print("\n3. Creating model...")
#     model = MDLMTransformer(
#         vocab_size=vocab_size,
#         d_model=768,
#         nhead=12,
#         num_layers=8,
#         dim_feedforward=3072,
#         dropout=0.1,
#         max_seq_len=128,
#     )
#     model = model.to(DEVICE)
#     total_params = sum(p.numel() for p in model.parameters())
#     print(f"   ✓ Parameters: {total_params:,}")

#     optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)


#     num_epochs = 10
#     print(f"\n4. Training for {num_epochs} epochs...")
#     print("="*80)

#     for epoch in range(num_epochs):
#         print(f"\nEpoch {epoch+1}/{num_epochs}")
#         print("-"*80)

#         model.train()
#         train_losses, train_accs = [], []

#         pbar = tqdm(train_loader, desc="Training", leave=True)
#         for batch in pbar:
#             loss, acc = train_step(
#                 model, batch, optimizer, tokenizer.mask_token_id, 'cosine', DEVICE
#             )
#             train_losses.append(loss)
#             train_accs.append(acc)
#             pbar.set_postfix({'loss': f"{loss:.4f}", 'acc': f"{acc:.3f}"})

#         avg_train_loss = np.mean(train_losses)
#         avg_train_acc  = np.mean(train_accs)


#         model.eval()
#         val_losses, val_accs = [], []

#         with torch.no_grad():
#             for batch in tqdm(val_loader, desc="Evaluation"):
#                 clean_tokens   = batch["input_ids"].to(DEVICE)
#                 attention_mask = batch["attention_mask"].to(DEVICE)
#                 keyword_ids    = batch["keyword_ids"].to(DEVICE)
#                 batch_size, seq_len = clean_tokens.shape

#                 # FIX: random t per example, just like training
#                 t = torch.rand(batch_size, 1, device=DEVICE)
#                 mask_probs = torch.tensor(
#                     [get_mask_prob(t_i.item(), 'cosine') for t_i in t], device=DEVICE
#                 ).view(batch_size, 1)

#                 noisy_input, mask_indices = create_masked_input(
#                     clean_tokens, tokenizer.mask_token_id, mask_probs, attention_mask
#                 )

#                 # FIX: no t passed to model
#                 logits = model(noisy_input, condition_tokens=keyword_ids)
#                 loss = F.cross_entropy(
#                     logits.view(-1, logits.size(-1)), clean_tokens.view(-1), reduction='none'
#                 )
#                 loss = loss.view(batch_size, seq_len)
#                 masked_loss = (loss * mask_indices.float()).sum() / (mask_indices.sum() + 1e-8)

#                 predictions = logits.argmax(dim=-1)
#                 correct     = (predictions == clean_tokens) & mask_indices
#                 accuracy    = correct.sum().float() / (mask_indices.sum() + 1e-8)

#                 val_losses.append(masked_loss.item())
#                 val_accs.append(accuracy.item())

#         avg_val_loss = np.mean(val_losses)
#         avg_val_acc  = np.mean(val_accs)

#         scheduler.step()

#         print(f"\n  Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.3f}")
#         print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.3f}")

#     print("\n" + "="*80)
#     print("5. Testing Generation")
#     print("="*80)

#     model.eval()
#     torch.save(model.state_dict(), "mdlm_model.pth")
#     print("   ✓ Model saved to mdlm_model.pth")

#     # 5 diverse keyword prompts covering different sentiments and aspects
#     test_keywords = [
#         "positive great food delicious",       # strong positive, food-focused
#         "negative bad service rude",           # strong negative, service-focused
#         "positive amazing atmosphere cozy",    # positive, ambiance-focused
#         "negative disappointing cold food",    # negative, food quality
#         "positive friendly staff excellent",   # positive, service-focused
#     ]

#     # 10 guidance scales: from no guidance → very strong guidance
#     guidance_scales = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.0, 10.0]

#     # Store results for the summary table at the end
#     results = {}

#     for keywords in test_keywords:
#         print(f"\n{'='*80}")
#         print(f"KEYWORDS: {keywords}")
#         print(f"{'='*80}")
#         results[keywords] = {}

#         for scale in guidance_scales:
#             text = sample_mdlm(
#                 model, tokenizer,
#                 keywords=keywords,
#                 n_steps=150,
#                 guidance_scale=scale,
#                 max_length=128,
#                 device=DEVICE
#             )
#             results[keywords][scale] = text
#             print(f"\n  guidance={scale:<5} | {text[:120]}{'...' if len(text) > 120 else ''}")

#     # ========================================================================
#     # 6. Results Summary Table
#     # ========================================================================
#     print("\n" + "="*80)
#     print("6. RESULTS SUMMARY")
#     print("="*80)
#     print(f"\n{'Keyword':<40} {'Scale':<8} {'Preview (first 80 chars)'}")
#     print("-"*80)

#     for keywords, scale_results in results.items():
#         keyword_short = keywords[:38]
#         for scale, text in scale_results.items():
#             preview = text[:78].replace('\n', ' ')
#             print(f"{keyword_short:<40} {scale:<8} {preview}")
#         print()

#     # ========================================================================
#     # 7. Guidance Scale Analysis
#     # ========================================================================
#     print("="*80)
#     print("7. GUIDANCE SCALE EFFECT (first keyword only)")
#     print("="*80)
#     first_keyword = test_keywords[0]
#     print(f"\nKeyword: '{first_keyword}'\n")
#     print(f"{'Scale':<8} | {'Observation'}")
#     print("-"*50)

#     scale_labels = {
#         0.5:  "BELOW 1 — pushes AWAY from keywords",
#         1.0:  "=1 — free generation, keywords ignored",
#         1.5:  "slight keyword influence",
#         2.0:  "moderate keyword influence",
#         2.5:  "noticeable keyword steering",
#         3.0:  "strong keyword influence (recommended)",
#         4.0:  "very strong steering",
#         5.0:  "heavy steering, may reduce fluency",
#         7.0:  "aggressive steering",
#         10.0: "maximum steering, likely repetitive",
#     }
#     for scale in guidance_scales:
#         print(f"  {scale:<6} | {scale_labels[scale]}")
#         print(f"         | {results[first_keyword][scale][:90]}")
#         print()

#     return model, tokenizer


# model, tokenizer = quick_test_mdlm()

'\nNOTE: This is commented out because it is the training process for the MDLM model\n'

In [18]:
# Start of evaluation
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({'mask_token': '[MASK]', 'pad_token': '<PAD>'})
vocab_size = len(tokenizer)

model = MDLMTransformer(
    vocab_size=vocab_size,
    d_model=768,
    nhead=12,
    num_layers=8,
    dim_feedforward=3072,
    max_seq_len=128
)

C:\Users\Merp\AppData\Local\Temp\ipykernel_58928\4039022250.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)


In [19]:
checkpoint = torch.load(mdlm_model_path, map_location="cpu")
model.load_state_dict(checkpoint)
model.eval()
model.to(DEVICE)

MDLMTransformer(
  (token_embedding): Embedding(50259, 768)
  (pos_embedding): Embedding(128, 768)
  (cond_proj): Linear(in_features=768, out_features=768, bias=True)
  (adaln): AdaLN(
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=False)
    (proj): Linear(in_features=768, out_features=1536, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-7): 8 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, 

In [20]:
eval_dataset = MDLMYelpDataset(
    test_prompted_ds,
    hf_tokenizer,
    max_length=128
)

In [21]:
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# eval_loader = DataLoader(
#     eval_dataset,
#     batch_size=16,
#     shuffle=False
# )

from torch.utils.data import Subset

small_eval = Subset(eval_dataset, range(1000))

eval_loader = DataLoader(
    small_eval,
    batch_size=16,
    shuffle=False
)

In [22]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if DEVICE == "cuda" else -1
)

prompt = "positive food tasty delicious"

print("Prompt:", prompt)
print("True label: 1")

generated = sample_mdlm(
    model,
    tokenizer,
    keywords=prompt,
    n_steps=150,
    max_length=128,
    guidance_scale=4,
    device=DEVICE
)

print("\nGenerated review:")
print(generated)

pred = sentiment_model(generated)[0]
print("\nSentiment prediction:", pred)

Device set to use cuda:0


Prompt: positive food tasty delicious
True label: 1

Generated review:
Wonderful vac came skeptical day!!! Delicious breakfast is delicious delicious delicious delicious price fresh price noodles. Pric comfortable Pric clean clean. Nice price workers polite decor leave Staff super busy so fast folks the menu welcomed felt filled very delicious good service to saving was right i qiter so friendly and smiles. Quick waiter, thank thank you are you work this pockets healthy thank thank thank honey coming today very warm not impressed Say experience was nice was welcomed away hot and fun experience. Delicious home and refreshing. trymy get back my disappears soon..... for the strong loved knowion eats the food he job clicks but mild comfortable before im Magn\ ''daid neighborhoods

Sentiment prediction: {'label': 'POSITIVE', 'score': 0.9995688796043396}


In [23]:
"""
NOTE: This is perplexity testing which will take too long for a simple show case
"""
# import torch
# import torch.nn.functional as F
# import math
# from tqdm import tqdm


# model.eval()

# total_loss = 0
# total_tokens = 0
# pad_id = gpt2_tokenizer.pad_token_id

# with torch.no_grad():
#     for batch in tqdm(eval_loader):

#         input_ids = batch["input_ids"].to(DEVICE)
#         keyword_ids = batch["keyword_ids"].to(DEVICE)

#         logits = model(input_ids, condition_tokens=keyword_ids)

#         loss = F.cross_entropy(
#             logits.view(-1, logits.size(-1)),
#             input_ids.view(-1),
#             ignore_index=pad_id,
#             reduction="sum"
#         )

#         total_loss += loss.item()
#         total_tokens += (input_ids != pad_id).sum().item()

# mdlm_loss = total_loss / total_tokens
# mdlm_perplexity = math.exp(mdlm_loss)

# print("MDLM Diffusion Evaluation Results")
# print("----------------------------------")
# print(f"Loss: {mdlm_loss:.4f}")
# print(f"Pseudo-Perplexity: {mdlm_perplexity:.2f}")

'\nNOTE: This is perplexity testing which will take too long for a simple show case\n'

In [24]:
"""
NOTE: From here on it's mainly just testing, so none of them will be run for the showcase
"""
# from tqdm import tqdm

# correct = 0
# total = 0

# N = 200

# for i in tqdm(range(N), desc="Evaluating MDLM sentiment"):

#     example = test_prompted_ds[i]
#     label = example["label"]
#     keywords = example["prompt"]

#     generated = sample_mdlm(
#         model,
#         hf_tokenizer,
#         keywords=keywords,
#         max_length=128,
#         device=DEVICE
#     )
#     print(generated)

#     pred = sentiment_model(generated)[0]["label"]
#     pred_label = 1 if pred == "POSITIVE" else 0

#     if pred_label == label:
#         correct += 1

#     total += 1

# accuracy = correct / total

# print("\nSentiment Accuracy:", accuracy)

"\nNOTE: From here on it's mainly just testing, so none of them will be run for the showcase\n"

In [25]:
# from transformers import GPT2LMHeadModel, AutoTokenizer

# model_path = "gpt2_yelp_finetuned"

# gpt2_tokenizer = AutoTokenizer.from_pretrained(model_path)
# gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

# gpt2_model = GPT2LMHeadModel.from_pretrained(model_path).to(DEVICE)
# gpt2_model.eval()

In [26]:
# One step training” sanity check

In [27]:
import torch

class GPT2EvalDataset(torch.utils.data.Dataset):
    def __init__(self, prompted_dataset, tokenizer, max_length=128):
        self.ds = prompted_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        review = self.ds[idx]["review"]

        enc = self.tokenizer(
            review,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return enc["input_ids"].squeeze()

In [28]:
from torch.utils.data import DataLoader, Subset

gpt2_eval_dataset = GPT2EvalDataset(test_prompted_ds, gpt2_tokenizer, max_length=128)

small_eval = Subset(gpt2_eval_dataset, range(1000))

gpt2_loader = DataLoader(
    small_eval,
    batch_size=16,
    shuffle=False
)

In [29]:
# import math
# from tqdm import tqdm

# total_loss = 0
# total_tokens = 0
# pad_id = gpt2_tokenizer.pad_token_id

# with torch.no_grad():
#     for input_ids in tqdm(gpt2_loader):

#         input_ids = input_ids.to(DEVICE)

#         outputs = gpt2_model(
#             input_ids=input_ids,
#             labels=input_ids
#         )

#         loss = outputs.loss

#         tokens = (input_ids != pad_id).sum()

#         total_loss += loss.item() * tokens.item()
#         total_tokens += tokens.item()

# gpt2_loss = total_loss / total_tokens
# gpt2_perplexity = math.exp(gpt2_loss)

# print("GPT-2 Evaluation Results")
# print("------------------------")
# print(f"Loss: {gpt2_loss:.4f}")
# print(f"Perplexity: {gpt2_perplexity:.2f}")

In [30]:
# from rouge_score import rouge_scorer
# from transformers import AutoTokenizer
# from tqdm.auto import tqdm

# def compute_rouge(references, hypotheses, use_stemmer=True):
#     scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=use_stemmer)
#     agg = {"rouge1": [], "rouge2": [], "rougeL": []}
#     for ref, hyp in zip(references, hypotheses):
#         s = scorer.score(ref, hyp)
#         for k in agg:
#             agg[k].append(s[k].fmeasure)
#     return {k: sum(v) / len(v) * 100 if v else 0.0 for k, v in agg.items()}

# model.to(DEVICE)
# tokenizer = AutoTokenizer.from_pretrained("gpt2")
# if tokenizer.pad_token_id is None:
#     tokenizer.pad_token_id = tokenizer.eos_token_id
# n_rouge = 10
# references = [test_prompted_ds[i]["review"] for i in range(n_rouge)]
# hypotheses = []
# for i in tqdm(range(n_rouge), desc="MDLM generate"):
#     prompt = test_prompted_ds[i]["prompt"]
#     hyp = sample_mdlm(model, tokenizer, keywords=prompt, device=DEVICE, n_steps=80, max_length=64)
#     hypotheses.append(hyp)
# scores = compute_rouge(references, hypotheses)
# print(f"ROUGE-1: {scores['rouge1']:.2f}")
# print(f"ROUGE-2: {scores['rouge2']:.2f}")
# print(f"ROUGE-L: {scores['rougeL']:.2f}")

In [31]:
# model.eval()

In [32]:
# model_path = "gpt2_yelp_finetuned"
# from transformers import AutoTokenizer, GPT2LMHeadModel
# gpt2_tokenizer = AutoTokenizer.from_pretrained(model_path)
# gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
# gpt2_model = GPT2LMHeadModel.from_pretrained(model_path).to(DEVICE)
# gpt2_model.eval()

# n_rouge_gpt2 = 10
# refs_gpt2 = [test_prompted_ds[i]["review"] for i in range(n_rouge_gpt2)]
# hyps_gpt2 = []
# for i in tqdm(range(n_rouge_gpt2), desc="GPT-2 generate"):
#     prompt = test_prompted_ds[i]["prompt"]
#     gen = generate_review_gpt2(prompt=prompt, max_new_tokens=100, do_sample=True)
#     hyps_gpt2.append(gen[0])
# scores_gpt2 = compute_rouge(refs_gpt2, hyps_gpt2)
# print("GPT-2 ROUGE")
# print(f"  ROUGE-1: {scores_gpt2['rouge1']:.2f}")
# print(f"  ROUGE-2: {scores_gpt2['rouge2']:.2f}")
# print(f"  ROUGE-L: {scores_gpt2['rougeL']:.2f}")

In [33]:
# gpt2_model.eval()

In [34]:
# from bert_score import score as bert_score

# P_mdlm, R_mdlm, F_mdlm = bert_score(hypotheses, references, lang="en")
# P_gpt2, R_gpt2, F_gpt2 = bert_score(hyps_gpt2, refs_gpt2, lang="en")

# print("BERTScore F1 (higher = better semantic overlap):")
# print(f"  MDLM:  {F_mdlm.mean().item():.4f}")
# print(f"  GPT-2: {F_gpt2.mean().item():.4f}")

We have more tests that are not included in this project ipynb, they are in metrics.py